This notebook is for modeling and evaluating span identification from the SemEval dataset. This is the first filter for propaganda, so we only want to filter out what we are confident is not propaganda (so high-sensitivity/high-recall). Then downstream, let the technique classification (TC) model handle the precision and pruning.

In [1]:
import os
from torch import nn, torch
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset
import evaluate

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Prevents hangs during data loading

In [3]:
#Set up paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "semeval_roberta_scanner"

In [4]:
#Check for GPU support
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Metal (MPS) for acceleration")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU. Training might be slow.")

Using Apple Metal (MPS) for acceleration


In [5]:
#Load span identification data
df_si = pd.read_csv(DATA_DIR / "semeval_si_cleaned.csv")
df_si.head()

,article_id,sentence_text,label
0,111111111,Next plague outbreak in Madagascar could be 's...,1
1,111111111,"""The next transmission could be more pronounce...",1
2,111111111,"An outbreak of both bubonic plague, which is s...",0
3,111111111,Madagascar has suffered bubonic plague outbrea...,0
4,111111111,The disease tends to make a comeback each hot ...,0


In [6]:
#Initialize the model
##Tried model_checkpoint = "bert-base-uncased", Best F1 score after 3 epochs: 0.26392
##model_checkpoint = "microsoft/deberta-v3-small", After 3 epoches, still getting gradient explosion every time, even after adjusting hyperparameters
#Roberta Best F1 score after 3 epochs: 0.304
model_checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

In [7]:
#Map the labels
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=3)
model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [8]:
#Use the 'sentence_text' and 'label' (binary) from cleaned SI data
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["sentence_text"], truncation=True, padding="max_length", max_length=128)
    labels = []
    for i, label in enumerate(examples["label"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) #Ignore special tokens
            elif label == 1:
                #If sentence is propaganda, label tokens as B/I
                label_ids.append(1 if word_idx == 0 else 2)
            else:
                label_ids.append(0) # O
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [9]:
#Tokenize and train-test split dataset
raw_dataset = Dataset.from_pandas(df_si).train_test_split(test_size=0.2)
tokenized_datasets = raw_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=raw_dataset["train"].column_names)

Map:   0%|          | 0/11252 [00:00<?, ? examples/s]

Map:   0%|          | 0/2813 [00:00<?, ? examples/s]

In [13]:
#Evaluate model
metric = evaluate.load("seqeval")
label_list = ["O", "B-Prop", "I-Prop"]

In [14]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [[label_list[p] for (p, l) in zip(pr, lb) if l != -100] for pr, lb in zip(predictions, labels)]
    true_labels = [[label_list[l] for (p, l) in zip(pr, lb) if l != -100] for pr, lb in zip(predictions, labels)]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {"precision": results["overall_precision"], "recall": results["overall_recall"], "f1": results["overall_f1"]}

In [15]:
#Tried without weighting before and was quickly overfitting
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Prioritize Recall: Propaganda classes (1, 2) weighted 7x more than background (0)
        weights = torch.tensor([1.0, 7.0, 7.0], device=model.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [18]:
#Set up training arguments with optimized hyperparameters
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16, # RoBERTa-base is lighter than DeBERTa; 16 should be stable
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    dataloader_pin_memory=False
)

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model(MODEL_DIR)

Epoch,Training Loss,Validation Loss
